# Collection — NASA Technical Reports Server (NTRS)

Source analyzed for the challenge *Create Health Monitoring Software for Astronauts on Space Missions* (NASA Space Apps 2026).

**Scope:** the *Life Sciences* division of the NASA STI category scheme, except Exobiology, plus *Space Radiation*.

| Category | Reason |
|---|---|
| Aerospace Medicine | Core of the challenge |
| Behavioral Sciences | Sleep, mood, isolation |
| Life Sciences (General) | Physiology and biology in spaceflight |
| Man/System Technology and Life Support | Human-system interaction, life support |
| Space Radiation | One of the five hazards of spaceflight |

Exobiology was excluded because it deals with life beyond Earth, not human health.

**Known API limitations**
- Pagination is blocked from record 10,000 onward (`page.from >= 10000` returns 400). Larger categories are sliced by publication date range.
- The category filter is case-sensitive. Man/System appears with two spellings, and both are collected.

## 1. Setup

In [1]:
import json
import os
import time
from datetime import date

import pandas as pd
import requests

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

In [2]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/space-apps-2026/ntrs"
RAW_DIR = os.path.join(DATA_DIR, "raw")
os.makedirs(RAW_DIR, exist_ok=True)

Mounted at /content/drive


In [3]:
BASE_URL = "https://ntrs.nasa.gov/api/citations/search"
PAGE_SIZE = 100
LIMITE_PAGINACAO = 10_000  # the API rejects page.from >= 10000
PAUSA = 0.3                # seconds between requests

CATEGORIAS = [
    "Aerospace Medicine",
    "Behavioral Sciences",
    "Space Radiation",
    "Life Sciences (General)",
    "Man/System Technology And Life Support",
    "Man/System Technology and Life Support",
]

## 2. API access functions

In [4]:
def requisitar(params, tentativas=4):
    """GET com novas tentativas. Retorna o JSON ou None se todas falharem."""
    for t in range(tentativas):
        try:
            resp = requests.get(BASE_URL, params=params, timeout=120)
            if resp.status_code == 400:
                return None  # parameter error, retrying will not help
            resp.raise_for_status()
            return resp.json()
        except requests.RequestException as e:
            print(f"    erro ({t + 1}/{tentativas}): {e}")
            time.sleep(3 * (t + 1))
    return None


def contar(params):
    data = requisitar({**params, "page.size": 1})
    return None if data is None else data["stats"]["total"]


def paginar(params, esperado):
    """Baixa todas as páginas de uma consulta com menos de 10.000 resultados."""
    registros = []
    inicio = 0
    while inicio < esperado and inicio < LIMITE_PAGINACAO:
        data = requisitar({**params, "page.size": PAGE_SIZE, "page.from": inicio})
        if data is None:
            print(f"    falha em from={inicio}, consulta interrompida")
            break
        pagina = data.get("results", [])
        if not pagina:
            break
        registros.extend(pagina)
        inicio += PAGE_SIZE
        time.sleep(PAUSA)
    return registros

### Slicing by date

For queries above 10,000 results, the publication date range is split in half repeatedly until each slice fits the limit.

Before using it, the function checks whether the API actually applies the date filter. If it does not, the category is collected up to the 10,000 limit and the problem is logged.

In [5]:
DATA_INICIO = date(1900, 1, 1)
DATA_FIM = date.today()


def filtro_data(ini, fim):
    return {"published.gte": ini.isoformat(), "published.lte": fim.isoformat()}


def filtro_data_funciona(cat, total_cat):
    meio = date(1990, 1, 1)
    antes = contar({"subjectCategory": cat, **filtro_data(DATA_INICIO, meio)})
    depois = contar({"subjectCategory": cat, **filtro_data(meio, DATA_FIM)})
    print(f"    teste do filtro de data: até 1990 = {antes}, desde 1990 = {depois}, total = {total_cat}")
    return antes is not None and depois is not None and antes < total_cat and depois < total_cat


def fatias_por_data(cat, ini, fim):
    """Divide [ini, fim] até cada fatia ter menos de LIMITE_PAGINACAO documentos."""
    n = contar({"subjectCategory": cat, **filtro_data(ini, fim)})
    if n is None or n == 0:
        return []
    if n < LIMITE_PAGINACAO or (fim - ini).days <= 1:
        return [(ini, fim, n)]
    meio = ini + (fim - ini) / 2
    return fatias_por_data(cat, ini, meio) + fatias_por_data(cat, meio, fim)

## 3. Collection

In [6]:
def coletar_categoria(cat):
    total_cat = contar({"subjectCategory": cat})
    print(f"{cat}: {total_cat} documentos na API")

    if total_cat < LIMITE_PAGINACAO:
        registros = paginar({"subjectCategory": cat}, total_cat)
        return registros, total_cat, "pagina_unica"

    if not filtro_data_funciona(cat, total_cat):
        print("    filtro de data ignorado pela API, coletando só os primeiros 10.000")
        registros = paginar({"subjectCategory": cat}, LIMITE_PAGINACAO)
        return registros, total_cat, "truncada"

    fatias = fatias_por_data(cat, DATA_INICIO, DATA_FIM)
    soma = sum(n for _, _, n in fatias)
    print(f"    {len(fatias)} fatias, somando {soma} documentos")
    if soma < total_cat:
        print(f"    {total_cat - soma} documentos sem data de publicação ficam fora das fatias")

    registros = []
    for ini, fim, n in fatias:
        print(f"    fatia {ini} a {fim}: {n}")
        registros.extend(paginar({"subjectCategory": cat, **filtro_data(ini, fim)}, n))
    return registros, total_cat, "fatiada"

In [7]:
def nome_arquivo(cat):
    return os.path.join(RAW_DIR, "cat_" + cat.replace("/", "-").replace(" ", "_") + ".json")


relatorio = []
for cat in CATEGORIAS:
    caminho = nome_arquivo(cat)
    if os.path.exists(caminho):
        with open(caminho) as f:
            salvo = json.load(f)
        print(f"já coletada: {cat} ({len(salvo['registros'])} registros)")
    else:
        registros, total_cat, modo = coletar_categoria(cat)
        salvo = {"categoria": cat, "total_api": total_cat, "modo": modo, "registros": registros}
        with open(caminho, "w") as f:
            json.dump(salvo, f)
        print(f"OK {cat}: {len(registros)} de {total_cat} ({modo})\n")
    relatorio.append({
        "categoria": salvo["categoria"],
        "total_api": salvo["total_api"],
        "coletados": len(salvo["registros"]),
        "modo": salvo["modo"],
    })

relatorio = pd.DataFrame(relatorio)
relatorio["cobertura_pct"] = (100 * relatorio["coletados"] / relatorio["total_api"]).round(1)
relatorio

Aerospace Medicine: 8278 documentos na API
OK Aerospace Medicine: 8278 de 8278 (pagina_unica)

Behavioral Sciences: 2079 documentos na API
OK Behavioral Sciences: 2079 de 2079 (pagina_unica)

Space Radiation: 9618 documentos na API
OK Space Radiation: 9618 de 9618 (pagina_unica)

Life Sciences (General): 11835 documentos na API
    teste do filtro de data: até 1990 = 2867, desde 1990 = 8218, total = 11835
    3 fatias, somando 10960 documentos
    875 documentos sem data de publicação ficam fora das fatias
    fatia 1900-01-01 a 1963-06-02: 5
    fatia 1963-06-02 a 1995-02-15: 4562
    fatia 1995-02-15 a 2026-11-01: 6393
OK Life Sciences (General): 10960 de 11835 (fatiada)

Man/System Technology And Life Support: 11121 documentos na API
    teste do filtro de data: até 1990 = 2764, desde 1990 = 7972, total = 11121
    3 fatias, somando 10641 documentos
    480 documentos sem data de publicação ficam fora das fatias
    fatia 1900-01-01 a 1963-06-02: 6
    fatia 1963-06-02 a 1995-02-15:

,categoria,total_api,coletados,modo,cobertura_pct
0,Aerospace Medicine,8278,8278,pagina_unica,100.0
1,Behavioral Sciences,2079,2079,pagina_unica,100.0
2,Space Radiation,9618,9618,pagina_unica,100.0
3,Life Sciences (General),11835,10960,fatiada,92.6
4,Man/System Technology And Life Support,11121,10641,fatiada,95.7
5,Man/System Technology and Life Support,986,986,pagina_unica,100.0


If the session drops during collection, just run the cell again. Categories already saved to Drive are skipped.
To redo a category, delete the corresponding file in `raw/`.

## 4. Consolidation

In [8]:
linhas = []
for cat in CATEGORIAS:
    with open(nome_arquivo(cat)) as f:
        salvo = json.load(f)
    for reg in salvo["registros"]:
        linhas.append({**reg, "_categoria_coleta": cat})

df_raw = pd.DataFrame(linhas)
print("registros coletados:", len(df_raw))

# the same document may appear in more than one category
cats_por_doc = df_raw.groupby("id")["_categoria_coleta"].apply(lambda s: sorted(set(s)))
df = df_raw.drop_duplicates("id").set_index("id")
df["_categorias_coleta"] = cats_por_doc
df = df.drop(columns="_categoria_coleta").reset_index()

print("documentos únicos:", len(df))
print("documentos em mais de uma categoria:", (df["_categorias_coleta"].str.len() > 1).sum())

registros coletados: 42562
documentos únicos: 41866
documentos em mais de uma categoria: 633


In [9]:
PARQUET_PATH = os.path.join(DATA_DIR, "ntrs_saude.parquet")

# nested fields become JSON text so parquet accepts them, and are decoded in the EDA
df_salvar = df.copy()
for col in df_salvar.columns:
    if df_salvar[col].apply(lambda v: isinstance(v, (list, dict))).any():
        df_salvar[col] = df_salvar[col].apply(lambda v: json.dumps(v) if isinstance(v, (list, dict)) else v)

df_salvar.to_parquet(PARQUET_PATH, index=False)
print("salvo em", PARQUET_PATH)
print(f"{os.path.getsize(PARQUET_PATH) / 1024**2:.1f} MB")

salvo em /content/drive/MyDrive/space-apps-2026/ntrs/ntrs_saude.parquet
48.7 MB


## 5. Next step

The EDA starts from `ntrs_saude.parquet`. When loading it, the nested columns (`authorAffiliations`, `publications`, `center`, `subjectCategories`, `keywords`, `downloads`) need `json.loads`.